# Notebook 05: Transformer for Text Classification

**Level:** Intermediate (bridges LSTM and Transformers)
**Time:** 45-60 minutes
**Prerequisites:** Notebooks 01 (Intent) or 02 (Sentiment)

## What You'll Learn

In notebooks 01-04, you built LSTM-based classifiers and generators. Now we take the logical next step: **replacing the LSTM with a Transformer encoder** for the same classification task.

By the end of this notebook you will:
- Understand how self-attention replaces recurrence
- Build a Transformer-based text classifier in pure MLX (no PyTorch)
- Compare LSTM vs Transformer performance on identical data
- Visualize positional encodings and compare training curves; notebook 06 shows attention weights

## Architecture Bridge

```
LSTM Classifier (Notebooks 01-04)    Transformer Classifier (This Notebook)
=============================        ===================================
Embedding                            Embedding + Positional Encoding
   |                                    |
LSTM (sequential, step-by-step)      TransformerEncoder (parallel)
   |                                  - Multi-Head Self-Attention
Last real token -> Linear               - Feed-Forward Network
   |                                  - Layer Normalization
Softmax -> Class                     Masked mean -> Linear -> Softmax
```

## Setup and Imports

In [ ]:
import sys
import os
import json
import numpy as np
from pathlib import Path

import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim

from mlx_nlp_utils import (
    IntentLSTM, SentimentLSTM,
    create_vocabulary, preprocess_text, texts_to_sequences, pad_sequences,
    train_model, predict_intent, predict_sentiment, evaluate_model,
    load_sample_sentiment_data, load_sample_intent_data,
    print_device_info
)

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print_device_info()
print(f'MLX version: {mx.__version__ if hasattr(mx, "__version__") else "installed"}')
print('Setup complete!')
from mlx.utils import tree_flatten


## Part 1: Build a Positional Encoding

Transformers process all tokens in parallel, so they need positional information injected into the embeddings. We implement sinusoidal positional encoding from [Vaswani et al. (2017)](https://arxiv.org/abs/1706.03762):

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
The table is stored as `_pos_encoding`, a non-trainable module attribute.


In [ ]:
def sinusoidal_positional_encoding(max_len, d_model):
    """Create sinusoidal positional encoding matrix."""
    pe = mx.zeros((max_len, d_model))
    position = mx.arange(0, max_len).reshape(-1, 1)
    div_term = mx.exp(mx.arange(0, d_model, 2) * (-mx.log(mx.array(10000.0)) / d_model))
    
    pe = pe.at[:, 0::2].add(mx.sin(position * div_term))
    pe = pe.at[:, 1::2].add(mx.cos(position * div_term[:d_model // 2]))
    return pe

# Visualize positional encoding
pe = sinusoidal_positional_encoding(100, 64)
pe_np = np.array(pe)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
im = ax1.imshow(pe_np[:50, :], aspect='auto', cmap='RdBu')
ax1.set_xlabel('Embedding Dimension'); ax1.set_ylabel('Position')
ax1.set_title('Positional Encoding (first 50 positions)')
plt.colorbar(im, ax=ax1)

# Show dot-product similarity between positions
sim = pe_np @ pe_np.T
ax2.imshow(sim[:50, :50], aspect='auto', cmap='viridis')
ax2.set_xlabel('Position'); ax2.set_ylabel('Position')
ax2.set_title('Position-Position Similarity (via dot product)')
plt.tight_layout()
plt.show()

## Part 2: Transformer Classifier Model

We build a classification model using MLX's built-in `TransformerEncoder`. The architecture:

1. **Embedding** — maps token IDs to dense vectors
2. **Positional Encoding** — adds position information
3. **TransformerEncoder** — stacks of self-attention + feed-forward layers
4. **Mean Pooling** — aggregate sequence into a single vector
5. **Classifier Head** — linear layer + softmax

In [ ]:
class TransformerClassifier(nn.Module):
    """Transformer-based text classifier using MLX's TransformerEncoder."""
    
    def __init__(self, vocab_size, d_model=64, num_heads=4, num_layers=2, 
                 num_classes=3, max_len=50, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.max_len = max_len
        
        self.embedding = nn.Embedding(vocab_size, d_model)
        self._pos_encoding = sinusoidal_positional_encoding(max_len, d_model)
        
        self.transformer_encoder = nn.TransformerEncoder(
            num_layers=num_layers,
            dims=d_model,
            num_heads=num_heads,
            mlp_dims=d_model * 4,
            dropout=dropout
        )
        
        self.classifier = nn.Linear(d_model, num_classes)
        self.dropout = nn.Dropout(dropout)
    
    def __call__(self, x):
        # x: (batch_size, seq_len)
        seq_len = x.shape[1]
        
        # Embedding + positional encoding
        embedded = self.embedding(x)  # (batch, seq_len, d_model)
        embedded = embedded + self._pos_encoding[:seq_len, :]
        
        # Boolean attention masks use True for allowed keys. Give empty
        # inputs one safe key to prevent fully masked softmax rows.
        valid = x != 0
        safe_keys = valid | ((mx.sum(valid, axis=1, keepdims=True) == 0)
                             & (mx.arange(seq_len)[None, :] == 0))
        encoded = self.transformer_encoder(embedded, mask=safe_keys[:, None, None, :])
        weights = valid[..., None].astype(encoded.dtype)
        pooled = mx.sum(encoded * weights, axis=1) / mx.maximum(mx.sum(weights, axis=1), 1)
        pooled = self.dropout(pooled)
        
        # Classifier head
        logits = self.classifier(pooled)  # (batch, num_classes)
        return logits

print('TransformerClassifier model defined!')

## Part 3: Load Data

We use the same sentiment & intent datasets from notebooks 01-02.

In [ ]:
# Load both sentiment and intent data for comparison
print('=' * 50)
print('SENTIMENT DATA')
print('=' * 50)
sent_texts, sent_labels, sent_vocab, sent_label2idx = load_sample_sentiment_data()
sent_label_names = list(sent_label2idx.keys())
print(f'\n{len(sent_texts)} examples, {len(sent_vocab)} vocabulary words')
print(f'Labels: {sent_label_names}')
print(f'Label distribution: {dict(zip(*np.unique(sent_labels, return_counts=True)))}')

print('\n' + '=' * 50)
print('INTENT DATA')
print('=' * 50)
intent_texts, intent_labels, intent_vocab, intent_label2idx = load_sample_intent_data()
intent_label_names = list(intent_label2idx.keys())
print(f'\n{len(intent_texts)} examples, {len(intent_vocab)} vocabulary words')
print(f'Labels: {intent_label_names}')
print(f'Label distribution: {dict(zip(*np.unique(intent_labels, return_counts=True)))}')

## Part 4: Preprocess & Pad

Convert text to integer sequences and pad to uniform length.

In [ ]:
def prepare_data(texts, labels, vocab, label2idx, max_len=30):
    """Convert texts to padded sequences and labels to indices."""
    # Build reverse mapping for <UNK> handling
    word_to_idx = {w: i for w, i in vocab.items()} if isinstance(vocab, dict) else {w: i for i, w in enumerate(vocab)}
    
    # Tokenize
    sequences = []
    for text in texts:
        tokens = text.lower().split()
        seq = [word_to_idx.get(t, word_to_idx.get('<UNK>', 1)) for t in tokens]
        sequences.append(seq)
    
    # Pad
    padded = np.zeros((len(sequences), max_len), dtype=np.int32)
    for i, seq in enumerate(sequences):
        length = min(len(seq), max_len)
        padded[i, :length] = seq[:length]
    
    # Labels to array
    if isinstance(labels[0], str):
        y = np.array([label2idx[l] for l in labels], dtype=np.int32)
    else:
        y = np.array(labels, dtype=np.int32)
    
    return padded, y

# Prepare sentiment data
X_sent, y_sent = prepare_data(sent_texts, sent_labels, sent_vocab, sent_label2idx)
print(f'Sentiment: X={X_sent.shape}, y={y_sent.shape}')

# Prepare intent data  
X_intent, y_intent = prepare_data(intent_texts, intent_labels, intent_vocab, intent_label2idx)
print(f'Intent:   X={X_intent.shape}, y={y_intent.shape}')

## Part 5: Reuse the Shared Mini-Batch Trainer

`train_transformer` wraps `train_model` to return its history. The shared
trainer already shuffles mini-batches, includes the final partial batch, clips
gradients, and compiles updates while capturing model, optimizer, and random
state. Both LSTM and Transformer metrics use evaluation mode.


In [ ]:
def train_transformer(model, X, y, epochs=30, lr=0.001, batch_size=32):
    """Use the shared compiled loop, including the final partial batch."""
    _, history = train_model(model, mx.array(X), mx.array(y), epochs=epochs,
                             learning_rate=lr, batch_size=batch_size)
    return history


## Part 6: Train Transformer vs LSTM

These runs compare training curves on the same toy data. They do not use a
held-out validation set, so they do not establish which architecture generalizes
better. Extend the split-and-vocabulary pattern from notebooks 01–02 to make
that comparison, keeping the training budget comparable.


In [ ]:
# ---- Train Transformer on Sentiment ----
print('=' * 60)
print('TRANSFORMER - Sentiment Classification')
print('=' * 60)

vocab_size_sent = len(sent_vocab)
transformer_sent = TransformerClassifier(
    vocab_size=vocab_size_sent,
    d_model=64,
    num_heads=4,
    num_layers=2,
    num_classes=len(sent_label_names),
    max_len=X_sent.shape[1],
    dropout=0.1
)

print(f'Parameters: {sum(p.size for _, p in tree_flatten(transformer_sent.parameters())):,}')
history_trans_sent = train_transformer(transformer_sent, X_sent, y_sent, epochs=80, lr=0.001, batch_size=16)
print(f'Final accuracy: {history_trans_sent["accuracy"][-1]:.4f}')

In [ ]:
# ---- Train LSTM on Sentiment (baseline) ----
print('\n' + '=' * 60)
print('LSTM BASELINE - Sentiment Classification')
print('=' * 60)

lstm_sent = SentimentLSTM(
    vocab_size=vocab_size_sent,
    embedding_dim=64,
    hidden_size=128,
    output_size=len(sent_label_names)
)

print(f'Parameters: {sum(p.size for _, p in tree_flatten(lstm_sent.parameters())):,}')

# Use the shared train_model from mlx_nlp_utils
lstm_sent_model, history_lstm_sent = train_model(
    lstm_sent,
    mx.array(X_sent),
    mx.array(y_sent),
    epochs=80,
    learning_rate=0.001
)
print(f'Final accuracy: {history_lstm_sent["accuracy"][-1]:.4f}')

In [ ]:
# ---- Train Transformer on Intent ----
print('\n' + '=' * 60)
print('TRANSFORMER - Intent Classification')
print('=' * 60)

vocab_size_intent = len(intent_vocab)
transformer_intent = TransformerClassifier(
    vocab_size=vocab_size_intent,
    d_model=64,
    num_heads=4,
    num_layers=2,
    num_classes=len(intent_label_names),
    max_len=X_intent.shape[1],
    dropout=0.1
)

history_trans_intent = train_transformer(transformer_intent, X_intent, y_intent, epochs=80, lr=0.001, batch_size=16)
print(f'Final accuracy: {history_trans_intent["accuracy"][-1]:.4f}')

## Part 7: Visualize the Comparison

Training curves side by side: Transformer vs LSTM.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Loss comparison
ax = axes[0]
ax.plot(history_trans_sent['loss'], label='Transformer (Sentiment)', linewidth=2)
ax.plot(history_lstm_sent['loss'], label='LSTM (Sentiment)', linewidth=2, linestyle='--')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Training Loss: Transformer vs LSTM')
ax.legend(); ax.grid(True, alpha=0.3)

# Accuracy comparison
ax = axes[1]
ax.plot(history_trans_sent['accuracy'], label='Transformer (Sentiment)', linewidth=2)
ax.plot(history_lstm_sent['accuracy'], label='LSTM (Sentiment)', linewidth=2, linestyle='--')
ax.plot(history_trans_intent['accuracy'], label='Transformer (Intent)', linewidth=2)
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.set_title('Accuracy Comparison')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 8: Key Differences — Why Transformers Win

| Property | LSTM | Transformer |
|----------|------|-------------|
| Processing | Sequential (one token at a time) | Parallel (all tokens at once) |
| Long-range dependencies | Fades over time (gradient issues) | Direct attention to any position |
| Training speed | Slower (can't parallelize) | Faster (matrix multiply dominated) |
| Model size | Fewer parameters | More parameters (but efficient ops) |
| Interpretability | Hidden state (opaque) | Attention weights (visualizable) |

**When to use LSTM:** Tiny models (<10K params), streaming data, very short sequences
**When to use Transformer:** Most NLP tasks, longer contexts, production systems

## Part 9: Interactive Prediction

Test both models on custom input.

In [ ]:
def predict_transformer(model, text, vocab, label_names, max_len=30):
    """Predict using transformer classifier."""
    word_to_idx = vocab if isinstance(vocab, dict) else {w: i for i, w in enumerate(vocab)}
    tokens = text.lower().split()
    seq = [word_to_idx.get(t, 1) for t in tokens]
    padded = np.zeros((1, max_len), dtype=np.int32)
    length = min(len(seq), max_len)
    padded[0, :length] = seq[:length]
    
    model.eval()
    X = mx.array(padded)
    logits = model(X)
    probs = mx.softmax(logits, axis=-1)[0]
    pred_idx = int(mx.argmax(probs))
    confidence = float(probs[pred_idx])
    return label_names[pred_idx], confidence

# Test on new examples
test_examples = [
    'This movie was absolutely fantastic',
    'I really hated the service',
    'It was okay I guess',
    'What time does the store open',
    'Turn on the kitchen light',
]

for text in test_examples:
    label_t, conf_t = predict_transformer(transformer_sent, text, sent_vocab, sent_label_names)
    label_l, conf_l = predict_sentiment(lstm_sent, text, sent_vocab, sent_label_names, max_len=30)
    print(f'"{text}"')
    print(f'  Transformer -> {label_t} ({conf_t:.1%}) | LSTM -> {label_l} ({conf_l:.1%})')
    print()

## Summary

You've now built a Transformer classifier from scratch in MLX! Key takeaways:

- **Positional encoding** replaces the sequential bias of LSTMs
- **Multi-head self-attention** lets every token attend to every other token
- **Masked mean pooling** excludes padding when building a classification vector
- Compare held-out results and measured runtime before choosing an architecture

**Next:** Head to **Notebook 06 (Attention Mechanism)** to understand the math behind self-attention, or **Notebook 07 (Build NanoGPT)** to implement the full decoder architecture.